# Imported Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import f1_score, accuracy_score, precision_score, confusion_matrix


Note: `DummyClassifier(strategy = "most_frequent")` predicts only one class. As the other 11 classes receive a precision value of 0/0 (undefined), this triggers an `UndefinedMetricWarning`. These 'undefined' values are then set to 0.0, which is expected. 

I've included this  filter to silence the `UndefinedMetricWarning`s which occur in cell 23.

In [ ]:
import warnings
import sklearn.exceptions

warnings.filterwarnings(
    "ignore",
    category = sklearn.exceptions.UndefinedMetricWarning
    )

In [ ]:
np.random.seed(42)

# Load audited data

In [ ]:
news = pd.read_csv("../datasets/audited_data.csv")

news.head(10)

# Split Dataset into Training and Test Data
First pre-processing step is to split the audited dataset into training and test data to avoid data leakage; I've decided to split using the standard 80/20 split.

In [ ]:
X = news[["title", "content"]]
y = news["type"]

X.head(10)

In [ ]:
X.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = 0.2)

In [ ]:
X_train.shape

In [ ]:
y_train.shape

# Minimal Preprocessing for Baseline Model
The baseline model I've chosen to use is `DummyClassifier`, with `strategy = uniform`; it will generate predictions at random, with each class having an equal probability of being chosen.


In [ ]:
dc = DummyClassifier(
    strategy = 'uniform',
    random_state = 42
    )


 Next, I set up stratified 10-fold cross-validation using `StratifiedKFold`, which will ensure that the proportion of article types is roughly equal across each fold.

In [ ]:
stratified_kfold_cv = StratifiedKFold(
    n_splits = 10,
    shuffle = True,
    random_state = 42
)

 Next I use `cross_validate()` to assess the baseline model on the training data. The model is fitted on nine folds and tested on the tenth. This process is repeated ten times so that the each fold is used for validation once. The mean of the ten Macro-F1 scores is 0.0130.

In [ ]:
metrics = {
    "accuracy": "accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro"
}

In [ ]:
results = cross_validate(
    dc,
    X_train,
    y_train,
    cv = stratified_kfold_cv,
    scoring = metrics,
    return_train_score = True
)

In [ ]:
for m in metrics:
    train_score = results[f"train_{m}"]
    test_score = results[f"test_{m}"]
    print(f"{m:16s} training data score = {train_score.mean():.4f} ± {train_score.std():.4f}")
    print(f"{m:16s} test data score = {test_score.mean():.4f} ± {test_score.std():.4f}")
    print(f"{m:16s} performance gap = {train_score.mean() - test_score.mean():.4f}")

-I used the fitted model to generate predictions (`y_pred`) based on the values in `X_test`, then compared them to the actual values (`y_test`).


In [ ]:
dc.fit(X_train, y_train)

In [ ]:
y_pred = dc.predict(X_test)

y_pred

As you can see, the resulting Macro-F1 score from the baseline model is pretty low; 0.0122. This is because the `DummyClassifier` is only making predictions based on the most common article type in the dataset. However, since I've taken a perfectly balanced sample (2500 records per class with 500 in the test set) it seems to be choosing a single random class to apply its predictions.

In [ ]:
test_accuracy_unnormalized = accuracy_score(
    y_test, 
    y_pred, 
    normalize = False
    )

test_accuracy = accuracy_score(
    y_test, 
    y_pred, 
    )

test_precision = precision_score(
    y_test,
    y_pred,
    average = "macro",
    zero_division = 0
)

test_f1 = f1_score(
    y_test,
    y_pred,
    average = "macro"
)

print(f"Baseline Macro-F1: {test_f1:.4f}")
print(f"Baseline Accuracy (raw counts): {test_accuracy_unnormalized:.4f}")
print(f"Baseline Accuracy (percentage): {test_accuracy:.4f}")
print(f"Baseline Precision (percentage): {test_precision:.4f}")

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels = dc.classes_ 
)

cm_df = pd.DataFrame(
    cm,
    index = dc.classes_,
    columns = dc.classes_
)

print(cm_df)